# View GSO-SLAM Results in Jupyter

This notebook loads the saved `point_cloud.ply` and `input.ply` outputs from a completed run and displays them inline with Open3D's Jupyter visualizer.

Open3D's Jupyter support is experimental and is designed for point cloud geometry. For Gaussian-splat PLY files, the notebook reads the stored vertex positions and shows them as point clouds.

If Open3D is missing in the notebook kernel, install it with `pip install open3d`.


In [ ]:
!pip install open3d


In [ ]:
from pathlib import Path

import open3d as o3d


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "experiments_bash").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")

REPO_ROOT = find_repo_root(Path.cwd())
RESULT_ROOT = REPO_ROOT / "experiments_bash" / "results" / "test"
print(f"Repo root: {REPO_ROOT}")
print(f"Result root: {RESULT_ROOT}")

In [ ]:
from pathlib import Path

import open3d as o3d


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "experiments_bash").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")

REPO_ROOT = find_repo_root(Path.cwd())
RESULT_ROOT = REPO_ROOT / "experiments_bash" / "results" / "test"
print(f"Repo root: {REPO_ROOT}")
print(f"Result root: {RESULT_ROOT}")

In [ ]:
def list_run_dirs(result_root: Path) -> list[Path]:
    run_dirs = set()
    for ply_path in result_root.glob("**/gs_map/point_cloud.ply"):
        run_dirs.add(ply_path.parent.parent)
    for ply_path in result_root.glob("**/gs_map/input.ply"):
        run_dirs.add(ply_path.parent.parent)
    return sorted(run_dirs)

run_dirs = list_run_dirs(RESULT_ROOT)
for idx, run_dir in enumerate(run_dirs):
    print(f"[{idx}] {run_dir.relative_to(REPO_ROOT)}")

RUN_NAME = "replica_room0_quality_codexfix"  # change this to the run you want
RUN_DIR = next((run_dir for run_dir in run_dirs if run_dir.name == RUN_NAME), None)
if RUN_DIR is None:
    raise FileNotFoundError(f"Could not find a run named {RUN_NAME!r} under {RESULT_ROOT}")

POINT_CLOUD_PLY = RUN_DIR / "gs_map" / "point_cloud.ply"
GS_SPLAT_PLY = RUN_DIR / "gs_map" / "input.ply"
print(f"Selected run: {RUN_DIR.relative_to(REPO_ROOT)}")
print(f"Point cloud: {POINT_CLOUD_PLY}")
print(f"GS splat:   {GS_SPLAT_PLY}")

In [ ]:
def load_point_cloud(path: Path, max_points: int | None = 500_000) -> o3d.geometry.PointCloud:
    pcd = o3d.io.read_point_cloud(str(path))
    if pcd.is_empty():
        raise RuntimeError(f"Open3D could not read any points from {path}")

    if max_points is not None and len(pcd.points) > max_points:
        ratio = max_points / float(len(pcd.points))
        pcd = pcd.random_down_sample(ratio)

    return pcd

def show_point_cloud(path: Path, title: str) -> None:
    pcd = load_point_cloud(path)
    print(title)
    print(pcd)
    try:
        o3d.visualization.draw(
            [pcd],
            title=title,
            width=1024,
            height=768,
            show_ui=True,
            point_size=2.0,
        )
    except Exception as exc:
        print(f"Open3D notebook viewer failed ({exc}); falling back to a standard viewer window.")
        o3d.visualization.draw_geometries([pcd], window_name=title, width=1024, height=768)

def describe_ply(path: Path) -> None:
    pcd = o3d.io.read_point_cloud(str(path))
    print(f"{path.name}: {len(pcd.points)} points")

In [ ]:
describe_ply(POINT_CLOUD_PLY)
show_point_cloud(POINT_CLOUD_PLY, "Gaussian map point cloud")

In [ ]:
describe_ply(GS_SPLAT_PLY)
show_point_cloud(GS_SPLAT_PLY, "Gaussian splat input cloud")

## Notes

- Open3D displays the xyz point data from both files; it does not render Gaussian-splat attributes in this notebook.
- If the point cloud is not centered well in the first view, use the mouse wheel to zoom and the right mouse button to pan.
- If you want to inspect another run, change `RUN_NAME` in the discovery cell.
- The saved files live under `experiments_bash/results/test/<run-name>/gs_map/`.